# LLM 04: Embeddings

Hands-on:
1. Embed with a local sentence-transformer (free, no API key)
2. Compute cosine similarity between related / unrelated sentences
3. Tiny retrieval demo over a 20-doc corpus
4. Matryoshka truncation: quality vs dimension
5. Exercises: query/passage asymmetry, OpenAI comparison

Deps: `pip install sentence-transformers numpy`

## 1. Embed Sentences with a Local Model

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')   # 384-dim, ~80MB

sentences = [
    'A dog runs in the park.',
    'A puppy is playing outside.',
    'The bank approved the loan.',
    'Interest rates rose yesterday.',
]

emb = model.encode(sentences, normalize_embeddings=True)
print('embedding shape:', emb.shape)
print('norm of first vector (should be 1.0):', np.linalg.norm(emb[0]))

## 2. Cosine Similarity Matrix

In [ ]:
sim = emb @ emb.T
print('cosine similarity matrix:')
print(sim.round(2))
print('\nexpected: (0,1) and (2,3) pairs should be high; cross-pairs low.')

## 3. Mini Retrieval Demo

In [ ]:
corpus = [
    'Python is a high-level programming language.',
    'PostgreSQL is a relational database.',
    'Transformers are a neural network architecture.',
    'Redis is an in-memory key-value store.',
    'PyTorch is a deep learning framework.',
    'Docker containerizes applications.',
    'BERT is a transformer-based model.',
    'Git tracks changes in source code.',
    'Kubernetes orchestrates containers.',
    'GPT is an autoregressive language model.',
]

corpus_emb = model.encode(corpus, normalize_embeddings=True)

def search(query, k=3):
    q = model.encode([query], normalize_embeddings=True)
    sim = (q @ corpus_emb.T)[0]
    top = np.argsort(-sim)[:k]
    return [(corpus[i], float(sim[i])) for i in top]

for q in ['tell me about neural networks', 'what database should I use', 'how to deploy apps']:
    print(f'\nQ: {q}')
    for doc, s in search(q):
        print(f'  {s:.3f}  {doc}')

## 4. Dimensionality Tradeoff: Truncation Quality

Simulate Matryoshka-style truncation by cutting the embedding and re-normalizing. Then measure how retrieval top-1 accuracy changes.

In [ ]:
# queries with their intended top match (manually labeled)
queries = [
    ('tell me about neural networks', 2),
    ('what database should I use', 1),
    ('how to deploy apps', 5),
    ('a programming language', 0),
    ('vector database? no wait', 3),
]

def truncate_norm(emb, d):
    e = emb[..., :d]
    return e / (np.linalg.norm(e, axis=-1, keepdims=True) + 1e-9)

for d in [384, 256, 128, 64, 32, 16]:
    corpus_t = truncate_norm(corpus_emb, d)
    hits = 0
    for q_text, gold in queries:
        q = truncate_norm(model.encode([q_text], normalize_embeddings=True), d)
        top = np.argmax(q @ corpus_t.T)
        hits += (top == gold)
    print(f'dim={d:>3}  top-1 accuracy: {hits}/{len(queries)}')

## 5. Visualize the Embedding Space

Project to 2D with PCA to see the clustering.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

labels = ['lang', 'db', 'ml', 'db', 'ml', 'ops', 'ml', 'tools', 'ops', 'ml']
colors = {'lang':'tab:blue', 'db':'tab:orange', 'ml':'tab:green', 'ops':'tab:red', 'tools':'tab:purple'}

pts = PCA(n_components=2).fit_transform(corpus_emb)
plt.figure(figsize=(8, 6))
for (x, y), lbl, txt in zip(pts, labels, corpus):
    plt.scatter(x, y, c=colors[lbl], s=60)
    plt.annotate(txt[:25], (x, y), fontsize=7)
plt.title('Corpus embeddings (PCA to 2D)')
plt.tight_layout()
plt.show()

## 6. Exercise: Query/Passage Asymmetry

Many state-of-the-art retrievers expect a prefix:
- BGE: `'query: ...'` for queries, `'passage: ...'` for documents
- E5: `'query: ...'` / `'passage: ...'`
- OpenAI: no prefixes

**Try it:** swap to `BAAI/bge-small-en-v1.5`, run the retrieval demo twice — once with prefixes and once without. Measure the top-1 accuracy difference. This single change often moves retrieval quality 5–15 points.

## 7. Exercise: OpenAI Comparison

If you have an OpenAI API key:
```python
from openai import OpenAI
client = OpenAI()
resp = client.embeddings.create(model='text-embedding-3-small', input=corpus, dimensions=256)
```
Repeat the retrieval evaluation. Compare top-1 accuracy on the 5 labeled queries against `all-MiniLM-L6-v2`. Note the cost: a few thousand tokens at current 2026 pricing is sub-cent.

## 8. Takeaways

- **Embeddings are dense, L2-normalized vectors** whose dot product measures semantic similarity.
- **Contextual models beat static ones** by a wide margin; sentence-transformers is the default toolkit.
- **Dimension matters for storage and speed.** Matryoshka models let you truncate with graceful degradation.
- **Query/passage asymmetry is the single most-missed RAG config.**
- **Evaluate on your data, not MTEB.**

Next: **05 — Pretraining Objectives**, where we compare how different model families are trained to predict tokens.